# 0. import libraries

In [1]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

# ── 2. circuit_tracer model for attribution (replaces HF model entirely) ─────
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend="transformerlens"
)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


# 1. getting attributes from each token step
based on attribute_demo  
**can skip to step 2, features are saved in** `demos\graph_files`

In [3]:
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 8192  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 256  # Batch size when attributing
offload = (
    "disk" if IN_COLAB else "cpu"
)  # Offload various parts of the model during attribution to save memory. Can be 'disk', 'cpu', or None (keep on GPU)
verbose = True  # Whether to display a tqdm progress bar and timing report

In [ ]:
# ── 3. Generation loop (same as before, but using ReplacementModel) ───────────
base_prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"]
STOP_IDS = {108, 235265, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out[0, -1, :].float()  # out is already the logits tensor
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
        [input_ids, torch.tensor([[next_id]])], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        max_feature_nodes=max_feature_nodes,   # ← missing
        batch_size=batch_size,
        offload=offload,                        # ← use the variable, not hardcoded "cpu"
        verbose=verbose,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

Phase 0: Precomputing activations and vectors


Generated: He saw a carrot and had to grab it


Precomputation completed in 0.79s
Found 15408 active features
Phase 1: Running forward pass
Forward pass completed in 17.76s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.6094
Will include 8192 of 15408 feature nodes
Input vectors built in 0.59s
Phase 3: Computing logit attributions
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\circuit_tracer\attribution\context_transformerlens.py:223: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  self._resid_activations[last_layer].backward(
Logit attributions completed in 5.89s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [02:33<00:00, 53.28it/s]
Feature attributions completed in 153.75s
Attribution completed in 178.89s


✓ step 00 → 'He'  saved as 'step-00-He'


Precomputation completed in 0.94s
Found 16293 active features
Phase 1: Running forward pass
Forward pass completed in 20.88s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5117
Will include 8192 of 16293 feature nodes
Input vectors built in 0.29s
Phase 3: Computing logit attributions
Logit attributions completed in 5.57s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [02:41<00:00, 50.63it/s]
Feature attributions completed in 161.82s
Attribution completed in 189.58s
Phase 0: Precomputing activations and vectors


...

✓ step 06 → ' to'  saved as 'step-06-to'


Precomputation completed in 0.69s
Found 22262 active features
Phase 1: Running forward pass
Forward pass completed in 28.34s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.8398
Will include 8192 of 22262 feature nodes
Input vectors built in 0.21s
Phase 3: Computing logit attributions
Logit attributions completed in 7.28s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [03:44<00:00, 36.47it/s]
Feature attributions completed in 224.62s
Attribution completed in 261.24s
Phase 0: Precomputing activations and vectors


✓ step 07 → ' grab'  saved as 'step-07-grab'


Precomputation completed in 1.05s
Found 23785 active features
Phase 1: Running forward pass
Forward pass completed in 28.10s
Phase 2: Building input vectors
Using 1 salient logits with cumulative probability 0.9883
Will include 8192 of 23785 feature nodes
Input vectors built in 0.21s
Phase 3: Computing logit attributions
Logit attributions completed in 7.82s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [03:54<00:00, 35.00it/s]
Feature attributions completed in 234.03s
Attribution completed in 271.30s


✓ step 08 → ' it'  saved as 'step-08-it'


In [ ]:

# extracting clerp descriptions and adding to graph files

import json, time, requests
from pathlib import Path

def fetch_clerp(layer, feature):
    url = f"https://www.neuronpedia.org/api/feature/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feature}"
    r = requests.get(url)
    if r.ok:
        data = r.json()
        descs = data.get("explanations", [])
        if descs:
            return descs[0].get("description", "")
    return ""

for json_path in sorted(Path("./graph_files").glob("*.json")):
    data = json.loads(json_path.read_text())
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        parts = node['node_id'].split('_')
        feat = parts[1]
        layer = node['layer']
        node['clerp'] = fetch_clerp(layer, feat)
        time.sleep(0.1)  # be nice to the API
    json_path.write_text(json.dumps(data))
    print(f"done: {json_path.name}")

done: step-00-He.json
done: step-01-saw.json
done: step-02-a.json
done: step-03-carrot.json
done: step-04-and.json
done: step-05-had.json
done: step-06-to.json
done: step-07-grab.json
done: step-08-it.json


In [ ]:
import json
from pathlib import Path

# load metadata from each poetry json and add to manifest
for json_path in sorted(Path("./graph_files").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


# 2. attribute graph visualization

In [10]:
import json
from pathlib import Path

# load metadata from each poetry json and add to manifest
for json_path in sorted(Path("./graph_files").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


In [15]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

http://localhost:8046/index.html


## 2.1 feature analysis based on circuit

In [2]:
import json
from pathlib import Path

def show_top_nodes(json_path, n=10):
    data = json.loads(Path(json_path).read_text())
    slug = data['metadata']['slug']
    
    # filter out logit nodes, sort by influence
    nodes = [n for n in data['nodes'] if not n['is_target_logit'] and n['influence'] is not None]
    nodes = sorted(nodes, key=lambda x: x['influence'], reverse=True)[:n]
    
    print(f"\n=== {slug} ===")
    for node in nodes:
        layer = node['layer']
        node_id_parts = node['node_id'].split('_')
        feat = node_id_parts[1]
        influence = node['influence']
        url = f"https://www.neuronpedia.org/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feat}"
        print(f"  L{layer}F{feat:>6}  influence={influence:.4f}  {url}")

for f in sorted(Path("./graph_files").glob("*.json")):
    show_top_nodes(f)


=== step-00-He ===
  L2F  6155  influence=0.8001  https://www.neuronpedia.org/gemma-2-2b/2-gemmascope-transcoder-16k/6155
  L4F  2540  influence=0.8000  https://www.neuronpedia.org/gemma-2-2b/4-gemmascope-transcoder-16k/2540
  L1F  1707  influence=0.7999  https://www.neuronpedia.org/gemma-2-2b/1-gemmascope-transcoder-16k/1707
  L1F  9553  influence=0.7999  https://www.neuronpedia.org/gemma-2-2b/1-gemmascope-transcoder-16k/9553
  L3F  5874  influence=0.7998  https://www.neuronpedia.org/gemma-2-2b/3-gemmascope-transcoder-16k/5874
  L3F  9033  influence=0.7997  https://www.neuronpedia.org/gemma-2-2b/3-gemmascope-transcoder-16k/9033
  L2F  8433  influence=0.7997  https://www.neuronpedia.org/gemma-2-2b/2-gemmascope-transcoder-16k/8433
  L2F  3589  influence=0.7996  https://www.neuronpedia.org/gemma-2-2b/2-gemmascope-transcoder-16k/3589
  L5F  8854  influence=0.7995  https://www.neuronpedia.org/gemma-2-2b/5-gemmascope-transcoder-16k/8854
  L2F  3247  influence=0.7995  https://www.neuronpedi

In [21]:
import json
from pathlib import Path
from collections import defaultdict

position_features = defaultdict(list)

for json_path in sorted(Path("./graph_files").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        token = data['metadata']['prompt_tokens'][node['ctx_idx']]
        if token == '\n':
            parts = node['node_id'].split('_')
            key = f"L{node['layer']}F{parts[1]}"
            position_features[key].append((json_path.stem, node['influence'], node['clerp']))

# show features appearing in 3+ steps, sorted by occurrence count
for feat, occurrences in sorted(position_features.items(), key=lambda x: -len(x[1])):
    if len(occurrences) >= 3:
        clerp = occurrences[0][2] or "no description"
        print(f"{feat}  ({len(occurrences)}/9 steps)  clerp: {clerp}")
        for step, influence, _ in occurrences:
            print(f"    {step}  influence={influence:.4f}")

L0F3850  (18/9 steps)  clerp: punctuation marks
    step-00-He  influence=0.4120
    step-00-He  influence=0.3882
    step-01-saw  influence=0.4222
    step-01-saw  influence=0.5832
    step-02-a  influence=0.4857
    step-02-a  influence=0.5280
    step-03-carrot  influence=0.4561
    step-03-carrot  influence=0.6505
    step-04-and  influence=0.4503
    step-04-and  influence=0.4326
    step-05-had  influence=0.4368
    step-05-had  influence=0.4317
    step-06-to  influence=0.4641
    step-06-to  influence=0.5084
    step-07-grab  influence=0.4576
    step-07-grab  influence=0.6396
    step-08-it  influence=0.4294
    step-08-it  influence=0.4814
L0F6625  (18/9 steps)  clerp:  single characters on a newline
    step-00-He  influence=0.6982
    step-00-He  influence=0.6009
    step-01-saw  influence=0.7185
    step-01-saw  influence=0.7812
    step-02-a  influence=0.7314
    step-02-a  influence=0.7272
    step-03-carrot  influence=0.7248
    step-03-carrot  influence=0.7970
    step

**Most promising (identified by Claude Sonnet 4.6):**

L2F629 — "marks verses, choruses, or segments of a musical composition" — active on \n across all 9 steps, consistently  
L8F13615 — "words and phrases related to poetry or creative writing with a bias toward personal anecdotes" — 17/9 steps  
L4F4619 — "lyrics of a specific song, possibly about love and uncertainty" — 17/9 steps  
L5F6651 — "sad and inquisitive poetry" — 9/9 steps, consistent influence ~0.6-0.78  
L8F12808 — "lines of poetry, especially those that express questioning, prayer, or lament" — 9/9 steps  
L13F4053 — "lines of rap lyrics" — 12/9 steps, high influence  
L3F14659 — "the language of rap lyrics" — 18/9 steps  
L14F15021 — "phrases that begin a line of poetry" — this one is particularly interesting given the context  
L9F11782 — "instances of the word 'poem' or descriptive phrases about things, often with a sense of place" — 5/9 steps  
L8F13396 — "mentions of poetic or musical art forms" — 5/9 steps, hits 0.8000 influence at step-07

# 3. intervention
based on intervention_demo

In [27]:
from collections import namedtuple
from functools import partial

import torch 

from circuit_tracer import ReplacementModel

# display functions
from circuit_tracer.utils.demo_utils import display_topk_token_predictions, display_generations_comparison

backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
model = ReplacementModel.from_pretrained("google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend=backend)


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [28]:
Feature = namedtuple('Feature', ['layer', 'pos', 'feature_idx'])

# a display function that needs the model's tokenizer
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

In [29]:
# L14F15021 = "phrases that begin a line of poetry"
supernode_features = [
    Feature(layer=14,pos=-1,feature_idx=15021),
]

intervention_tuples = [(*supernode_feature, 0.0) for supernode_feature in supernode_features]

In [30]:
s = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"
with torch.inference_mode():
    original_logits, _  = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

Token,Probability,Distribution
He,0.152,15.2%
But,0.152,15.2%
And,0.072,7.2%
The,0.044,4.4%
Then,0.039,3.9%
Token,Probability,Distribution
He,0.155,15.5%
But,0.137,13.7%
And,0.073,7.3%
but,0.039,3.9%


In [35]:
from circuit_tracer.utils.demo_utils import display_generations_comparison

s = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# suppress both candidate features at the newline position (-1)
intervention_tuples = [
    (14, -1, 15021, 0.0), # L14F15021 - phrases that begin a line of poetry
]

pre = [model.feature_intervention_generate(s, [], do_sample=False, verbose=False)[0]]
post = [model.feature_intervention_generate(s, intervention_tuples, do_sample=False, verbose=False)[0]]

display_generations_comparison(s, pre, post)